In [2]:
from sctoolbox.utils.jupyter import bgcolor, _compare_version

# change the background of input cells
bgcolor("PowderBlue", select=[2, 6, 9])

nb_name = "01_assembling_anndata.ipynb"

_compare_version(nb_name)

# 01 - Assembling or loading anndata object
<hr style="border:2px solid black"> </hr>

## 1 - Description
This notebook is dedicated to loading or creating an anndata object suitable for the subsequent analysis pipeline. The anndata object is prepared and finally stored as a `.h5ad` file. Based on the available data files there are multiple options to create the anndata object. To satisfy all and especially ATAC-related functionalities indexes are prepared to hold barcodes and feature coordinates.

### Available options:
#### 1. `.h5ad` file: 
Choose this option if you have one or more `.h5ad` file(s). The file could be provided by a preprocessing pipeline, a public dataset or a preceeding analysis.

#### 2. Convert from R object:
This option should be used if the data was processed using R. This can either be a `.rds` or `.robj` file.
   
#### 3. .mtx, barcode.tsv, [regions.tsv]
Choose this option if you have the count matrix in `.mtx` format, a file containing the barcodes (`*_barcodes.tsv`) and an optional file containing the regions (`*_regions.tsv`). Use this option for cases with the aforementioned three files available e.g. from a public dataset.

___________

## 2 - Setup

In [3]:
import pandas as pd
# sctoolbox modules 
import sctoolbox
import sctoolbox.utils as utils

sctoolbox.settings.settings_from_config("config.yaml", key="01")

with pd.option_context("display.max.rows", None, "display.max_colwidth", None):
    display(utils.general.get_version_report(report="versions.yml"))

[WARNING] Log file '../logs/01_log.txt' already exists. The file will be overwritten since 'overwrite_log' is set to True.


,Name,Version
0,Python,"3.12.12 | packaged by conda-forge | (main, Oct 22 2025, 23:25:55) [GCC 14.3.0]"
1,numpy,2.3.5
2,pandas,2.3.3
3,sctoolbox,NA


___

## 3 - Read in data
<hr style="border:2px solid black"> </hr>

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [4]:
# Choose one option

# For option 1: The path to an existing .h5ad file.
# A list or dict for multiple files e.g. {"rep_1": "1.h5ad", "rep_2": "2.h5ad"}.
path_h5ad = ""

# For option 2: This is the path to the R-related (.rds, .robj) file
# A list or dict for multiple files e.g. {"rep_1": "1.rds", "rep_2": "2.rds"}.
path_rds = ""
# Optional: Provide name of Seurat layer to be stored in anndata.
# When reading multiple Seurat files set to string to get the same layer from every object
# or set to list or dict to get specified layers from each object.
# In the second case it must be the same datatype as path_rds.
layer = None

# For option 3: Directory containing .mtx, barcodes.tsv and optionally regions.tsv
path_mtx = "../../../data/ATAC/filtered_peak_bc_matrix/"

batch_label = None  # Name of the column added to adata.obs only if Option 1 or 4 above is a dict.

----------------

In [5]:
if sum(map(lambda x: x != "", [path_h5ad, path_rds, path_mtx])) != 1:
    del path_h5ad, path_mtx, path_rds
    raise ValueError("Please set only one of the above variables. Adjust the cell above and re-run.")

### 3.1 - Option 1: Read from h5ad

In [6]:
if path_h5ad:
    adata = utils.assemblers.from_h5ad(path_h5ad, label=batch_label, report="01_assembly.png")

___

### 3.2 - Option 2: Convert from Seurat to anndata object

In [7]:
# Converting from Seurat to anndata object
if path_rds:
    adata = utils.assemblers.from_R(rds_file=path_rds, label=batch_label, layer=layer, report="01_assembly.png")

____

### 3.3 - Option 3: .mtx, barcode.tsv, [regions.tsv]

In [8]:
# adjust in case of different naming schemes for any of the input files

mtx = '*matrix.mtx*'  # pattern for the file that contains counts
barcodes = '*barcodes.tsv*'  # pattern for the file that contains barcode information
variables = '*peaks.bed*'  # pattern for the optional file that contains variable information

var_index = None # column in the variables file to use as .var index. For multiple coordinates: None

In [9]:
if path_mtx:
    adata = utils.assemblers.from_mtx(path_mtx, mtx=mtx, barcodes=barcodes, variables=variables, var_error=False, var_index=var_index, report="01_assembly.png")

/mnt/workspace2/hschult/.conda/sctoolbox/lib/python3.12/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


[INFO] Reading files: 1 of 1 
[WARNING] Barcodes file is one more than expected. Trying to fix by enabling the header.
[WARNING] Variables file is one more than expected. Trying to fix by enabling the header.


/mnt/workspace2/hschult/.conda/sctoolbox/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


____

## 4 - Prepare anndata
<hr style="border:2px solid black"> </hr>

Rename or remove observation (`.obs`) and variable (`.var`) columns, and layers as needed and format their indices. After this step the index of `.var` holds the feature coordinates and `.obs` the cell barcodes.

The AnnData object contains all information regarding the dataset at hand. It is commonly stored as a `.h5ad` file. As it is central to this single cell analysis here is a broad overview of its key attributes:

Attributes are accessed with `AnnData.<attribute_name>` e.g. `adata.obs`.

| Attribute | Description |
|-----------|-------------|
|`obs`|A table (pandas DataFrame) containing information to each observation. Observations often refer to individual barcodes or wells and are interpreted as cells.|
|`var`|A table (pandas DataFrame) containing information to each variable. Variables are the measurements taken for each cell and are e.g. genes (RNA) or peaks (ATAC).|
|`X`|A (sparse) matrix containing the actual measurement values. It has the shape `obs x var`. It is the main matrix used for calculations and contains read counts interpreted as gene expression (RNA) or chromatin openness (ATAC). Depending on the analysis progress this may be normalized or corrected.|
|`layers`|A dict-like structure containing additional named matrices. This can be used to store different versions of the matrix e.g. 'raw' and 'normalized'.|
|`raw`|A special attribute containing a raw representation of the dataset. A number of Scanpy functions use this implicitly when available, thus we recommend storing the raw data within the `layers` to avoid confusion.|

For more information visit the [AnnData Read the Docs](https://anndata.readthedocs.io/en/latest/index.html).

In [10]:
import pandas as pd

with pd.option_context('display.max_rows', 5,'display.max_columns', None):
    display(adata.obs)
    display(adata.var)
    display(f"AnnData.raw: {'Yes' if adata.raw else 'No'}")
    display(f"AnnData layers: {list(adata.layers.keys())}")

,filename,rel_path
index,,
AAACGAAAGAGAGGTA-1,matrix.mtx,.
AAACGAAAGCAGGAGG-1,matrix.mtx,.
...,...,...
TTTGTGTTCTGATCTT-1,matrix.mtx,.
TTTGTGTTCTTACTCA-1,matrix.mtx,.


,0,1,2
index,,,
0,chr1,9772,10660
1,chr1,180712,181178
...,...,...,...
165432,KI270713.1,29676,30535
165433,KI270713.1,36913,37813


'AnnData.raw: No'

'AnnData layers: []'

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [11]:
## 1. Modify existing columns

# .obs column names that should be deleted
drop_obs = ["filename", "rel_path"]

# .obs column names that should be changed. E.g. "old_name": "New Name"
rename_obs = {}

# .var column names that should be deleted
drop_var = []

# .var column names that should be changed. E.g. "old_name": "New Name"
rename_var = {'0': 'chr', '1': 'start', '2': 'end'}

# Whether to remove the AnnData.raw. Either a boolean or a string to save raw in `adata.layers[<name>]`
allow_raw = False

# adjust layer names. E.g. "old_name": "New Name"
rename_layer = {}

# save adata.X in a layer
keep_X = None

# replace adata.X with another layer
replace_X = None

# layer names that should be kept. 'all' to keep everything or a list of names
keep = 'all'


## 2. ATAC specific anndata properties
# The following settings are used to format the index and coordinate columns 

# Column name(s) of adata.var containing peak location data.
# Either a single column (str) or a list of three columns (['chr', 'start', 'end']) or None to use index.
coordinate_cols = ['chr', 'start', 'end']

# when formatting the index, should the prefix be removed
remove_var_index_prefix = True

# provide a name to save the original index, if None it will be overwritten
keep_original_index = None

# regex to format the index
coordinate_regex = r"chr[0-9XYM]+[\_\:\-]+[0-9]+[\_\:\-]+[0-9]+"

In [12]:
# manually add cell information
adata.obs["sample"] = "sample 1"
adata.obs["disease state"] = "healthy"
adata.obs["tissue"] = "blood"
adata.obs["selected cell types"] = "PBMCs"

_________

### 4.1 - Rename and delete columns/ layers 

In [13]:
# change obs
obs = adata.obs.copy()

obs.drop(columns=drop_obs, inplace=True)
obs.rename(columns=rename_obs, errors='raise', inplace=True)

# change var
var = adata.var.copy()

var.drop(columns=drop_var, inplace=True)
var.rename(columns=rename_var, errors='raise', inplace=True)

# apply changes to adata
adata.obs = obs
adata.var = var

# edit adata layers
utils.adata.tidy_layers(adata, allow_raw=allow_raw, rename=rename_layer, keep_X=keep_X, replace_X=replace_X, keep=keep)

___

### 4.2 - Format anndata indices
Set the cell barcode as `.obs` index and peak location as `.var` index.

In [14]:
adata = utils.assemblers.prepare_atac_anndata(adata,
                                              coordinate_cols=coordinate_cols,
                                              remove_var_index_prefix=remove_var_index_prefix,
                                              keep_original_index=keep_original_index,
                                              coordinate_regex=coordinate_regex)

[INFO] formatting index
[INFO] formatting adata.var index from coordinate columns.
[INFO] formatting coordinate columns
[INFO] The coordinate columns are: ['chr', 'start', 'end']
setting adata.obs.index.name = barcode


________

## 5 - Saving the anndata object

In [15]:
# Overview of loaded adata
display(adata)

AnnData object with n_obs × n_vars = 10246 × 165434
    obs: 'sample', 'disease state', 'tissue', 'selected cell types'
    var: 'chr', 'start', 'end'
    uns: 'sctoolbox'

In [16]:
# Saving the data
adata_output = "anndata_1.h5ad"
utils.adata.save_h5ad(adata, adata_output, report=["01_dataset_overview.md", "01_obs.png", "01_var.png"])

[INFO] The adata object was saved to: ../adatas/anndata_1.h5ad


In [17]:
sctoolbox.settings.close_logfile()